# Run the full AgentDojo experiment on Kaggle

Enable **Internet** and select a **GPU** in the notebook settings. The default smoke test runs all four conditions on the first configured suite. It selects one user task in each condition and one injection task in each malicious condition. AgentDojo may also evaluate that injection task as a standalone baseline. Set `TASK_FRACTION` to a decimal such as `0.25` to run 25% of each suite's user and injection tasks, rounded up separately. Fraction mode takes precedence over the smoke test. A fraction of `1.0` runs every task. With `TASK_FRACTION = None`, set `SMOKE_TEST = False` for a full run.

Select a branch that contains the updated `full_experiment.py` with `--task-fraction`. Push the updated scripts to that branch before running this notebook. The final cell prints utility and security summaries and saves the console log and task traces under `/kaggle/working`.

In [ ]:
REPOSITORY_URL = "https://github.com/Luke-Wojciuch/Large-Language-Models-Security-and-Privacy-Project.git"
BRANCH = "qwen-judge"
SMOKE_TEST = True
TASK_FRACTION = None  # Example: 0.25 runs the first 25% of each task list.
MODEL = "qwen3.5:9b"

## Check the GPU and prepare the Kaggle session

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path
from urllib.error import URLError
from urllib.request import Request, urlopen

subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
    check=True,
)

scratch = Path("/kaggle/temp")
scratch.mkdir(parents=True, exist_ok=True)

## Install Ollama, download Qwen, and run a GPU check

In [ ]:
ollama = shutil.which("ollama")
if ollama is None:
    installer = scratch / "install-ollama.sh"
    subprocess.run(["curl", "-fsSL", "https://ollama.com/install.sh", "-o", str(installer)], check=True)
    subprocess.run(["sh", str(installer)], check=True)
    ollama = shutil.which("ollama")
    if ollama is None:
        raise RuntimeError("Ollama was installed but its executable is missing from PATH.")

address = "http://localhost:11434"
ollama_env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_MODELS=str(scratch / "ollama-models"))
server_log = Path("/kaggle/working/ollama.log")

def ollama_json(endpoint, payload=None, timeout=5):
    request = Request(
        address + endpoint,
        data=None if payload is None else json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urlopen(request, timeout=timeout) as response:
        return json.load(response)

try:
    version = ollama_json("/api/version")
except URLError:
    with server_log.open("a") as log:
        server = subprocess.Popen(
            [ollama, "serve"], env=ollama_env, stdout=log, stderr=subprocess.STDOUT
        )
    deadline = time.monotonic() + 60
    while True:
        try:
            version = ollama_json("/api/version")
            break
        except URLError as error:
            if server.poll() is not None or time.monotonic() >= deadline:
                raise RuntimeError(f"Ollama failed to start. See {server_log}.") from error
            time.sleep(1)

print(f"Ollama version: {version['version']}", flush=True)
# Download the model into Ollama's cache. Reruns reuse verified local layers.
print(f"Running ollama pull {MODEL}. The first download may take several minutes.", flush=True)
try:
    subprocess.run([ollama, "pull", MODEL], env=ollama_env, check=True)
except subprocess.CalledProcessError as error:
    raise RuntimeError(
        f"Could not download {MODEL}. Check Kaggle Internet access and the Ollama output above."
    ) from error
installed = ollama_json("/api/tags")["models"]
model_info = next((entry for entry in installed if entry["name"] == MODEL), None)
if model_info is None:
    raise RuntimeError(f"{MODEL} is missing from the Ollama model list.")
print(f"Downloaded {MODEL}, digest {model_info['digest']}", flush=True)

print(f"Loading {MODEL} and generating a test response.", flush=True)
check = ollama_json(
    "/api/chat",
    {
        "model": MODEL,
        "messages": [{"role": "user", "content": "Reply with the word ready."}],
        "stream": False,
        "think": False,
    },
    timeout=180,
)
reply = check["message"]["content"].strip()
if not reply:
    raise RuntimeError(f"{MODEL} returned an empty response.")
print(f"Model response: {reply}")

loaded = next((entry for entry in ollama_json("/api/ps")["models"] if entry["name"] == MODEL), None)
if loaded is None or loaded["size_vram"] <= 0:
    raise RuntimeError("Ollama is not running the model on the GPU. Check the accelerator setting and server log.")
subprocess.run([ollama, "ps"], env=ollama_env, check=True)

## Clone the repository and install dependencies

In [ ]:
project = Path(tempfile.mkdtemp(prefix="agentdojo-", dir=scratch))
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY_URL, str(project)],
    check=True,
)
subprocess.run(["git", "log", "-1", "--oneline"], cwd=project, check=True)

launcher = project / "full_experiment.py"
if not launcher.is_file() or "--task-fraction" not in launcher.read_text():
    raise RuntimeError("Push the updated full_experiment.py to the selected branch and rerun this cell.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt"],
    cwd=project,
    check=True,
)
configured_model = subprocess.check_output(
    [sys.executable, "-c", "from full_experiment import EXPERIMENT_OPTIONS; print(EXPERIMENT_OPTIONS['model'])"],
    cwd=project,
    text=True,
).strip()
if configured_model != MODEL:
    raise RuntimeError(f"The cloned launcher uses {configured_model!r}. Expected {MODEL!r}.")

## Run the experiment and print summary statistics

In [ ]:
if TASK_FRACTION is not None:
    mode = "fraction"
else:
    mode = "smoke" if SMOKE_TEST else "full"
output = Path(tempfile.mkdtemp(prefix=f"agentdojo-{mode}-", dir="/kaggle/working"))
console_log = output / "experiment.log"
log_root = output / "runs"
command = [sys.executable, "-u", "full_experiment.py", "--log-root", str(log_root)]
if TASK_FRACTION is not None:
    command.extend(["--task-fraction", str(TASK_FRACTION)])
elif SMOKE_TEST:
    command.append("--smoke-test")

print(f"Running {mode} experiment. Logs: {output}", flush=True)
with console_log.open("w", encoding="utf-8") as log, subprocess.Popen(
    command,
    cwd=project,
    env=dict(os.environ, PYTHONIOENCODING="utf-8"),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()

if process.returncode:
    raise RuntimeError(f"Experiment exited with code {process.returncode}. See {console_log}.")
print(f"Experiment console log: {console_log}")
print(f"Per-task traces: {log_root}")